# Paper 14 · Retrieval-Augmented Generation

**Citation:** Patrick Lewis et al., “Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks” (2020).

**Paper:** https://arxiv.org/abs/2005.11401

> **Scale gap:** We reproduce retrieval-plus-context on a tiny local corpus using TF-IDF. We evaluate retrieval separately from generation.

## Mathematical Framework

Before reproducing the paper experimentally, work through:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 04 · Statistics & Likelihood](../../math/04_statistics_likelihood.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 09 · PCA, SVD & Kernels](../../math/09_pca_svd_kernels.ipynb)
- [Math 11 · Attention & Transformer Mathematics](../../math/11_attention_transformers.ipynb)

Your explanation should connect the paper's empirical claim to its **mathematical objective, representation, assumptions, and optimization/statistical argument**.

## Before you read
1. What can retrieval change without changing generator weights?
2. Why should retriever quality be measured separately?
3. What failure occurs when the correct evidence never enters context?

## Central claim
A parametric generator can be combined with non-parametric retrieval so answers can use external evidence not stored solely in model parameters.

## Tiny corpus and transparent retriever

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
from coursekit.experiments import Experiment
experiment = Experiment('paper-14_rag', {'bootstrap_seed': SEED, 'scope': 'educational mechanism demonstration', 'note': 'Original notebook may use additional explicit seeds; source hash records the exact experiment.'}, source='papers/notebooks/14_rag.ipynb')
experiment.capture_figures()

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
docs=[
 "Gradient descent updates model parameters opposite the loss gradient.",
 "PCA projects centered data onto orthogonal directions of high variance.",
 "A random forest averages many randomized decision trees.",
 "Self-attention forms weighted combinations of values using query-key similarity.",
 "RAG retrieves external evidence and gives it to a generator as context.",
 "LoRA learns low-rank updates while keeping base weights frozen.",
]
queries=[
 ("How does RAG obtain knowledge outside model weights?",{4}),
 ("What does PCA preserve?",{1}),
 ("What are queries and keys used for?",{3}),
 ("How does LoRA reduce trainable parameters?",{5}),
]
vec=TfidfVectorizer().fit(docs); M=vec.transform(docs)
def search(q,k=3):
    s=(M@vec.transform([q]).T).toarray().ravel()
    return np.argsort(s)[::-1][:k].tolist()

## Retrieval metrics

In [ ]:
rankings=[search(q,5) for q,_ in queries]
def recall_at_k(k):
    return np.mean([len(set(r[:k])&rel)/len(rel) for r,(_,rel) in zip(rankings,queries)])
def mrr():
    vals=[]
    for r,(_,rel) in zip(rankings,queries):
        vals.append(next((1/i for i,d in enumerate(r,1) if d in rel),0))
    return np.mean(vals)
table=pd.DataFrame({"k":[1,3,5],"Recall@k":[recall_at_k(k) for k in [1,3,5]]})
display(table); print("MRR",mrr())
for (q,_),r in zip(queries,rankings):
    print("\nQUERY:",q,"\nTOP:",docs[r[0]])

## Retrieval ablation

In [ ]:
rng=np.random.default_rng(0)
random_rankings=[rng.permutation(len(docs))[:5].tolist() for _ in queries]
good=rankings; rankings=random_rankings
print("random Recall@1",recall_at_k(1),"random MRR",mrr())
rankings=good
print("retriever Recall@1",recall_at_k(1),"retriever MRR",mrr())

### Generation activity
Write a tiny function that answers only from the top-k retrieved text and returns document IDs. Then deliberately ask an unanswerable question and design an abstention rule.

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this paper?
2. What was actually new?
3. What evidence did your notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which idea survived into modern systems?
6. What would you test next?

## Evidence export

Figures and numeric diagnostics are captured. Explicit metrics use `experiment.log(variant, seed, metrics)`. Use `run_trials` for paired-seed ablations. An empty metrics table or `not_run` ablation is incomplete evidence, not success. Interpretations remain your work.

In [ ]:
print('Evidence directory:', experiment.finish(globals()))